In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import spectral_norm
from s2flow.utils import get_device

class ResidualDenseBlock(nn.Module):
    """
    Residual Dense Block (RDB) for RRDB.
    """
    def __init__(self, num_feat=64, num_growth=32):
        super(ResidualDenseBlock, self).__init__()
        self.conv1 = nn.Conv2d(num_feat, num_growth, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_feat + num_growth, num_growth, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_feat + 2 * num_growth, num_growth, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_feat + 3 * num_growth, num_growth, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_feat + 4 * num_growth, num_feat, 3, 1, 1)

        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

        # initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
                m.weight.data *= 0.1
                if m.bias is not None:
                    m.bias.data.zero_()

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        # Empirically, residual scaling is used
        return x5 * 0.2 + x


class RRDB(nn.Module):
    """
    Residual in Residual Dense Block (RRDB).
    """
    def __init__(self, num_feat, num_growth=32):
        super(RRDB, self).__init__()
        self.rdb1 = ResidualDenseBlock(num_feat, num_growth)
        self.rdb2 = ResidualDenseBlock(num_feat, num_growth)
        self.rdb3 = ResidualDenseBlock(num_feat, num_growth)

    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        return out * 0.2 + x


class RRDBNet(nn.Module):
    """
    The Generator for Real-ESRGAN.
    Standard ESRGAN architecture: RRDBNet.
    """
    def __init__(self, in_channels=3, out_channels=3, num_feat=64, num_block=23, num_growth=32, scale=4):
        super(RRDBNet, self).__init__()
        self.scale = scale
        
        # 1. First convolution
        self.conv_first = nn.Conv2d(in_channels, num_feat, 3, 1, 1)

        # 2. Main Body (RRDB blocks)
        self.body = nn.Sequential(*[
            RRDB(num_feat, num_growth) for _ in range(num_block)
        ])
        
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)

        # 3. Upsampling
        self.upsample = nn.Sequential(
            nn.Upsample(scale_factor=scale, mode='nearest'),
            nn.Conv2d(num_feat, num_feat, 3, 1, 1),
            nn.LeakyReLU(negative_slope=0.2, inplace=True)
        )
        
        # 4. Final convolution
        self.conv_last_1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last_2 = nn.Conv2d(num_feat, out_channels, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        feat = self.conv_first(x)
        body_feat = self.conv_body(self.body(feat))
        feat = feat + body_feat
        feat = self.upsample(feat)
        feat = self.conv_last_1(feat)
        feat = self.lrelu(feat)
        out = self.conv_last_2(feat)
        return out


class UNetDiscriminatorSN(nn.Module):
    """
    U-Net Discriminator with Spectral Normalization.
    As described in Real-ESRGAN paper[cite: 4307, 4310].
    """
    def __init__(self, num_in_ch=3, num_feat=64, skip_connection=True):
        super(UNetDiscriminatorSN, self).__init__()
        self.skip_connection = skip_connection
        norm = spectral_norm

        self.conv0 = norm(nn.Conv2d(num_in_ch, num_feat, 3, 1, 1))

        self.conv1 = norm(nn.Conv2d(num_feat, num_feat * 2, 4, 2, 1, bias=False))
        self.conv2 = norm(nn.Conv2d(num_feat * 2, num_feat * 4, 4, 2, 1, bias=False))
        self.conv3 = norm(nn.Conv2d(num_feat * 4, num_feat * 8, 4, 2, 1, bias=False))

        # Up-sampling
        self.conv4 = norm(nn.Conv2d(num_feat * 8, num_feat * 4, 3, 1, 1, bias=False))
        self.conv5 = norm(nn.Conv2d(num_feat * 4, num_feat * 2, 3, 1, 1, bias=False))
        self.conv6 = norm(nn.Conv2d(num_feat * 2, num_feat, 3, 1, 1, bias=False))

        # Extra convs
        self.conv7 = norm(nn.Conv2d(num_feat, num_feat, 3, 1, 1, bias=False))
        self.conv8 = norm(nn.Conv2d(num_feat, num_feat, 3, 1, 1, bias=False))

        self.conv9 = nn.Conv2d(num_feat, 1, 3, 1, 1)

    def forward(self, x):
        x0 = F.leaky_relu(self.conv0(x), negative_slope=0.2, inplace=True)
        
        # Down
        x1 = F.leaky_relu(self.conv1(x0), negative_slope=0.2, inplace=True)
        x2 = F.leaky_relu(self.conv2(x1), negative_slope=0.2, inplace=True)
        x3 = F.leaky_relu(self.conv3(x2), negative_slope=0.2, inplace=True)

        # Up
        x3 = F.interpolate(x3, scale_factor=2, mode='bilinear', align_corners=False)
        x4 = F.leaky_relu(self.conv4(x3), negative_slope=0.2, inplace=True)

        if self.skip_connection:
            x4 = x4 + x2
        
        x4 = F.interpolate(x4, scale_factor=2, mode='bilinear', align_corners=False)
        x5 = F.leaky_relu(self.conv5(x4), negative_slope=0.2, inplace=True)

        if self.skip_connection:
            x5 = x5 + x1

        x5 = F.interpolate(x5, scale_factor=2, mode='bilinear', align_corners=False)
        x6 = F.leaky_relu(self.conv6(x5), negative_slope=0.2, inplace=True)

        if self.skip_connection:
            x6 = x6 + x0

        # Output
        out = self.conv9(F.leaky_relu(self.conv8(F.leaky_relu(self.conv7(x6), negative_slope=0.2, inplace=True)), negative_slope=0.2, inplace=True))
        
        return out

In [ ]:
from typing import Any, Dict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from s2flow.models import UNetDiscriminatorSN
from s2flow.data.pca import PCAConvLayer
from s2flow
from collections import OrderedDict

class PerceptualLoss(nn.Module):
    """
    Perceptual loss using VGG19 conv1-5 features before activation.
    Weights: {0.1, 0.1, 1, 1, 1} as specified in Real-ESRGAN paper[cite: 4323].
    """
    def __init__(self, config: Dict[str, Any]):
        super(PerceptualLoss, self).__init__()
        pca_conv = PCAConvLayer(config)
        vgg = torchvision.models.vgg19(weights=torchvision.models.VGG19_Weights.DEFAULT)
        # Layers: conv1_2, conv2_2, conv3_4, conv4_4, conv5_4
        # Indices in features: 2, 7, 16, 25, 34 (approx, depends on implementation, verifying standard)
        # Standard VGG19 feature indices: 
        # '2': conv1_2, '7': conv2_2, '12': conv3_2, '21': conv4_2, '30': conv5_2
        # Paper uses "conv1...conv5 feature maps... before activation". 
        # We extract features after the convolution but before ReLU.
        
        self.features = nn.ModuleList(list(vgg.features))
        self.layer_indices = {
            'conv1': 2, 
            'conv2': 7, 
            'conv3': 16, # conv3_4
            'conv4': 25, # conv4_4
            'conv5': 34  # conv5_4
        }
        # Weights from paper
        self.weights = {'conv1': 0.1, 'conv2': 0.1, 'conv3': 1.0, 'conv4': 1.0, 'conv5': 1.0}
        
        for param in self.parameters():
            param.requires_grad = False
            
    def forward(self, x, y):
        loss = 0
        x_feat = x
        y_feat = y
        
        current_layer = 0
        for name, index in sorted(self.layer_indices.items(), key=lambda item: item[1]):
            # Run layers up to the target index
            for i in range(current_layer, index):
                x_feat = self.features[i](x_feat)
                y_feat = self.features[i](y_feat)
            
            # The target convolution layer
            x_feat = self.features[index](x_feat)
            y_feat = self.features[index](y_feat)
            
            loss += self.weights[name] * F.l1_loss(x_feat, y_feat)
            current_layer = index + 1
            
        return loss

class RealESRGANTrainer(SRTrainer):
    """
    Trainer implementing the Real-ESRGAN training pipeline.
    Includes GAN training loop, U-Net Discriminator, and specific losses.
    """
    def _init_task_specific(self):
        # 1. Initialize Discriminator
        self.discriminator = UNetDiscriminatorSN(num_in_ch=3, num_feat=64).to(self.device)
        logger.debug("Initialized UNetDiscriminatorSN with Spectral Norm.")

        # 2. Losses [cite: 4322]
        # L1 Loss weight: 1.0
        # Perceptual Loss weight: 1.0
        # GAN Loss weight: 0.1
        self.cri_pix = nn.L1Loss().to(self.device)
        self.cri_perceptual = PerceptualLoss().to(self.device)
        self.cri_gan = nn.BCEWithLogitsLoss().to(self.device)
        
        self.loss_weights = {'l1': 1.0, 'percep': 1.0, 'gan': 0.1}

        # 3. Optimizers
        # Real-ESRGAN uses slightly different LR for G (1e-4) than Pretraining (2e-4) [cite: 4320]
        # We assume self.init_lr is set correctly in config.
        self.optimizer_G = self.optimizer # Alias the one created in BaseTrainer
        self.optimizer_D = torch.optim.AdamW(
            self.discriminator.parameters(), 
            lr=self.init_lr, 
            weight_decay=self.config.get('hyperparameters', {}).get('weight_decay', 0)
        )
        
        # Override scheduler to be a list or handle both manually
        self.scheduler_G = self.lr_scheduler
        self.scheduler_D = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer_D, T_max=self.num_epochs - self.warmup_epochs
        ) # Simplification, assuming same scheduler logic
        
        self.loss_name = 'combined_loss'

    def fit(self, train_dataloader, val_dataloader=None):
        # Overriding fit to handle GAN training loop
        logger.info("Starting Real-ESRGAN training loop (GAN).")
        for epoch in range(self.current_epoch, self.num_epochs + 1):
            self.current_epoch = epoch
            self.metrics_tracker.reset_epoch()
            
            self._run_gan_epoch(train_dataloader)
            
            if val_dataloader:
                self._run_phase('val', val_dataloader)
                
            self.metrics_tracker.finalize_epoch()
            self.scheduler_G.step()
            self.scheduler_D.step()
            self.save_checkpoint()
            self.save_metrics()
            self.save_model()

    def _run_gan_epoch(self, dataloader):
        self.model.train()
        self.discriminator.train()
        
        with tqdm(dataloader, desc=f"Train Epoch {self.current_epoch}", unit="bt") as pbar:
            for i, batch in enumerate(pbar):
                # Data
                lq, gt = map(lambda x: x.to(self.device), batch)
                
                # -------------------------
                # 1. Optimize Discriminator
                # -------------------------
                for p in self.discriminator.parameters():
                    p.requires_grad = True
                self.optimizer_D.zero_grad()

                # Fake
                fake_hr = self.model(lq)
                pred_d_fake = self.discriminator(fake_hr.detach())
                loss_d_fake = self.cri_gan(pred_d_fake, torch.zeros_like(pred_d_fake))
                
                # Real
                pred_d_real = self.discriminator(gt)
                loss_d_real = self.cri_gan(pred_d_real, torch.ones_like(pred_d_real))
                
                loss_d = (loss_d_real + loss_d_fake) * 0.5
                loss_d.backward()
                self.optimizer_D.step()

                # -------------------------
                # 2. Optimize Generator
                # -------------------------
                for p in self.discriminator.parameters():
                    p.requires_grad = False
                self.optimizer_G.zero_grad()
                
                # Pixel Loss
                l_g_pix = self.loss_weights['l1'] * self.cri_pix(fake_hr, gt)
                
                # Perceptual Loss
                l_g_percep = self.loss_weights['percep'] * self.cri_perceptual(fake_hr, gt)
                
                # GAN Loss
                pred_g_fake = self.discriminator(fake_hr)
                l_g_gan = self.loss_weights['gan'] * self.cri_gan(pred_g_fake, torch.ones_like(pred_g_fake))
                
                loss_g = l_g_pix + l_g_percep + l_g_gan
                loss_g.backward()
                self.optimizer_G.step()
                
                # Update Metrics
                self.metrics_tracker.update_batch(
                    'train', 
                    loss_g.detach(), 
                    fake_hr.detach(), 
                    gt,
                    additional_metrics={
                        'loss_d': loss_d.item(),
                        'loss_g_pix': l_g_pix.item(),
                        'loss_g_gan': l_g_gan.item()
                    }
                )
                
                pbar.set_postfix({
                    'l_g': f'{loss_g.item():.3f}',
                    'l_d': f'{loss_d.item():.3f}'
                })

    def save_checkpoint(self):
        # Override to save discriminator state
        state = {
            'model': self.model.state_dict(),
            'discriminator': self.discriminator.state_dict(),
            'optimizer_G': self.optimizer_G.state_dict(),
            'optimizer_D': self.optimizer_D.state_dict(),
            'epoch': self.current_epoch,
            'metrics': self.metrics_tracker.to_dict()
        }
        torch.save(state, self.checkpoint_path)

    def load_checkpoint(self):
        # Override to load discriminator
        if not self.checkpoint_path.exists(): return
        ckpt = torch.load(self.checkpoint_path, map_location=self.device)
        self.model.load_state_dict(ckpt['model'])
        if 'discriminator' in ckpt:
            self.discriminator.load_state_dict(ckpt['discriminator'])
        self.optimizer_G.load_state_dict(ckpt['optimizer_G'])
        if 'optimizer_D' in ckpt:
            self.optimizer_D.load_state_dict(ckpt['optimizer_D'])
        self.current_epoch = ckpt['epoch'] + 1
        logger.info(f"Resumed from epoch {self.current_epoch}")